In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=api_key)

print(client.api_key)

In [ ]:
fp = open("../data/audio/베트남여행.mp3", "rb")

transcriptions = client.audio.transcriptions.create(model="whisper-1", file=fp)
print(transcriptions)
print(transcriptions.text)

In [ ]:
print(dir(transcriptions))

In [ ]:
summary = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are summarize pro"},
        {
            "role": "user",
            "content": f"Please summarize the following : \n{transcriptions.text}",
        },
    ],
)

print(summary.choices[0].message.content)

In [ ]:
def transcribe_audio(file_path):
    fp = open(file_path, "rb")
    transcription = client.audio.transcriptions.create(file=fp, model="whisper-1")
    fp.close()
    return transcription.text


def transcribe_and_summarize(transcription_text):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You are helpful assistant that summarizes text",
            },
            {
                "role": "user",
                "content": f"다음 내용을 간단히 요약해줘 : \n{transcription_text}",
            },
        ],
    )

    return response.choices[0].message.content


transcription_text = transcribe_audio("../data/audio/베트남여행.mp3")

print("오디오로부터 추출된 텍스트 : ", transcription_text)
print()
print("요약본 : ", transcribe_and_summarize(transcription_text))

In [ ]:
response = client.audio.speech.create(
    input="너를 처음 만난 그 날은 드미트리 메드베데프 러시아 대통령 재임 시절 확률분포표상에는 있을 수 없는 청 단풍잎이 우거진 붉은 수수밭에서 수사슴 수사에 붙은 수수료가 얼마인지 알아보기 위해 간 그곳이었지 너의 얼굴은 마치 페니실린 살균 항균작용을 한 듯 하얗고 입술은 붉은 팥 풋 팥죽처럼 고왔어 그 시절 박남정 춤을 추며 안흥팥찐빵을 먹던 네 모습은 마치 내게 접근금지라고 말하는 듯했어 하지만 이내 우리는 강력접착제처럼 철수 책상 철 책상에 앉아 서로를 액자 속 사진 속에 홍합을 나눠 먹으며 그렇게 그렇게 행복해했지 하지만 이내 우리는 강력접착제처럼 철수 책상 철 책상에 앉아 서로를 액자 속 사진 속에 왕밤빵을 나눠 먹으며 행복해했지",
    model="tts-1",
    voice="nova",
)

print(response)

with open("../output/out.mp3", "wb") as fp:
    fp.write(response.read())

In [ ]:
from openai import OpenAI
from playsound import playsound
from IPython.display import Audio

prompt = input("prompt : ")

response = client.chat.completions.create(
    model="gpt-4o-mini", messages=[{"role": "system", "content": prompt}]
)

resp = response.choices[0].message.content
print("gpt : ", resp)

response = client.audio.speech.create(
    input=resp, model="tts-1", voice="nova", speed=1.5
)

with open("../output/playsound.mp3", "wb") as fp:
    fp.write(response.read())


Audio("../output/playsound.mp3", autoplay=True)

In [ ]:
from openai import OpenAI
import requests
import base64

response = client.images.generate(
    prompt="초 천재가 개발하는 이미지 만들어줘",
    model="gpt-image-2",
    size="1024x1024",
    quality="medium",
    n=1,
)


image_data = base64.b64decode(response.data[0].b64_json)
with open("../output/genimage.png", "wb") as fp:
    fp.write(image_data)

print("이미지가 output/genimage.png 경로에 성공적으로 저장되었습니다.")


In [ ]:
from openai import OpenAI
import base64  # 파이썬 표준 라이브러리


def encode_image(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


image_base64 = encode_image("../data/image/명함.png")  # 여기에 본인의 로컬 이미지 경로

data_url = f"data:image/png;base64,{image_base64}"

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "이미지의 글자 추출해줄래"},
                {"type": "image_url", "image_url": {"url": data_url}},
            ],
        }
    ],
)

print(response.choices[0].message.content)

In [ ]:
from openai import OpenAI
import base64
import requests

# 1. 위키미디어 이미지 다운로드 (User-Agent 헤더 추가로 차단 회피)
url = "https://postfiles.pstatic.net/MjAyNjAzMTZfMjU1/MDAxNzczNjI4NzQ3NzE5.HNIZDEh-URp_V6NiumkTPNJpixf3Cn8GtZfk8GdYnm0g.DIZ0AxmDF6bjFAyrkoqqCmz0gzDxGTDwKha7g9bI-vQg.JPEG/IMG%EF%BC%BF5983.jpg?type=w966"
headers = {"User-Agent": "Mozilla/5.0"}
img_response = requests.get(url, headers=headers)

# 2. 이미지 바이너리를 Base64 문자열로 변환
base64_image = base64.b64encode(img_response.content).decode("utf-8")

# 3. Vision API 요청 (MIME 프리픽스 data:image/jpeg;base64, 포함)
prompt = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "이 이미지에 대해 설명해줘."},
            {
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"},
            },
        ],
    }
]

response = client.chat.completions.create(model="gpt-4o-mini", messages=prompt)
print(response.choices[0].message.content)